# AndinaLog 03B | Eventos de flota | Tratamiento

Este notebook consume exclusivamente la salida diagnosticada y conserva los valores originales junto a los tratados.

## Decisiones

- Normalización de texto y fechas a formato uniforme en hora de Bolivia.
- `N/D` se transforma en nulo analítico con bandera.
- `reconocido` faltante permanece nulo y queda señalado.
- Las copias posteriores permanecen en cuarentena final.

In [ ]:
from pathlib import Path
import json
import re
import sys
import pandas as pd

ENTORNO = "auto"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "datasets/AndinaLog_03B_Bronce/andinalog_flota_eventos.json").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets/AndinaLog_03B_Bronce/andinalog_flota_eventos.json").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ = encontrar_raiz()
RUTA_BRONZE = RAIZ / "datasets/AndinaLog_03B_Bronce/andinalog_flota_eventos.json"

RUTA_DIAGNOSTICADO = RAIZ / "proyecto-integrador/01_diagnostico/andinalog_flota_eventos/salidas/andinalog_flota_eventos_diagnosticado.csv"
SALIDAS = RAIZ / "proyecto-integrador/02_tratamiento/andinalog_flota_eventos/salidas"
df = pd.read_csv(RUTA_DIAGNOSTICADO, encoding="utf-8-sig", keep_default_na=False)
originales = ["sistema", "fecha_exportacion", "camion_id", "evento_id", "timestamp", "tipo", "severidad", "valor_lectura", "reconocido", "umbral_temp_cabina_c", "geocerca_radio_km", "ultimo_mantenimiento"]
base = df[["fila_bronze", *originales, "en_cuarentena"]].copy()
for c in originales:
    base[f"{c}_original"] = base[c]

# Normalizaciones justificadas; la hora sin zona explícita se interpreta como Bolivia.
base["sistema"] = base["sistema"].astype("string").str.strip()
base["camion_id"] = base["camion_id"].astype("string").str.strip().str.upper()
base["evento_id"] = base["evento_id"].astype("string").str.strip().str.upper()
base["tipo"] = base["tipo"].astype("string").str.strip().str.upper()
base["severidad"] = base["severidad"].astype("string").str.strip().str.title()
ts_iso = pd.to_datetime(base["timestamp"], format="ISO8601", errors="coerce")
ts_local = pd.to_datetime(base["timestamp"], format="%d/%m/%Y %H:%M", errors="coerce")
base["timestamp_bolivia"] = ts_iso.fillna(ts_local).dt.strftime("%Y-%m-%d %H:%M:%S")
base["fecha_exportacion_bolivia"] = pd.to_datetime(base["fecha_exportacion"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
base["valor_lectura_no_disponible"] = base["valor_lectura"].astype("string").str.strip().eq("N/D")
base["valor_lectura_numerico"] = pd.to_numeric(base["valor_lectura"], errors="coerce")
base["valor_lectura_unidad_declarada"] = False
rec = base["reconocido"].astype("string").str.strip().str.lower()
base["reconocido_faltante"] = rec.isin(["", "<na>", "nan", "none"])
base["reconocido"] = rec.map({"true": True, "false": False}).astype("boolean")
base["umbral_temp_cabina_c"] = pd.to_numeric(base["umbral_temp_cabina_c"], errors="coerce")
base["geocerca_radio_km"] = pd.to_numeric(base["geocerca_radio_km"], errors="coerce")
base["ultimo_mantenimiento"] = pd.to_datetime(base["ultimo_mantenimiento"], errors="coerce").dt.strftime("%Y-%m-%d")

cuarentena_final = base.loc[base["en_cuarentena"]].copy()
silver = base.loc[~base["en_cuarentena"]].copy()
silver = silver.drop(columns=["en_cuarentena"])
silver["dias_desde_ultimo_mantenimiento"] = (
    pd.to_datetime(silver["timestamp_bolivia"]) - pd.to_datetime(silver["ultimo_mantenimiento"])
).dt.total_seconds().div(86400)
columnas_orden = ["fila_bronze", *[c for c in originales if c not in {"timestamp", "fecha_exportacion", "valor_lectura"}],
                  "timestamp_bolivia", "fecha_exportacion_bolivia", "valor_lectura_numerico",
                  "valor_lectura_no_disponible", "valor_lectura_unidad_declarada", "reconocido_faltante",
                  "dias_desde_ultimo_mantenimiento", *[f"{c}_original" for c in originales]]
silver = silver[columnas_orden]
metricas = {"filas_entrada": len(df), "filas_silver": len(silver), "filas_cuarentena_final": len(cuarentena_final),
            "valor_lectura_nd_en_silver": int(silver["valor_lectura_no_disponible"].sum()),
            "reconocido_faltante_en_silver": int(silver["reconocido_faltante"].sum())}
reporte = pd.DataFrame([{"metrica": k, "valor": v} for k, v in metricas.items()])
assert len(df) == len(silver) + len(cuarentena_final)
assert len(silver) == 184 and len(cuarentena_final) == 8
assert not silver["evento_id"].duplicated().any()
SALIDAS.mkdir(parents=True, exist_ok=True)
silver.to_csv(SALIDAS / "andinalog_flota_eventos_silver.csv", index=False, encoding="utf-8-sig")
cuarentena_final.to_csv(SALIDAS / "andinalog_flota_eventos_cuarentena_final.csv", index=False, encoding="utf-8-sig")
reporte.to_csv(SALIDAS / "andinalog_flota_eventos_reporte_calidad.csv", index=False, encoding="utf-8-sig")
print(metricas)
print(silver.head(3).to_string())


## Uso posterior

Silver puede agregarse temporalmente por `camion_id` y enlazarse con IoT usando solo eventos estrictamente anteriores a cada lectura.